In [10]:
import os

for folder in ['bronze', 'silver', 'gold']:
    os.makedirs(folder, exist_ok=True)


In [11]:
import pandas as pd

# 1. Load raw files without cleaning (handling ';' delimiter and latin-1 encoding)
users_df = pd.read_csv('BX-Users.csv', sep=';', encoding='latin-1', on_bad_lines='skip')
books_df = pd.read_csv('BX-Books.csv', sep=';', encoding='latin-1', on_bad_lines='skip')
ratings_df = pd.read_csv('BX-Book-Ratings.csv', sep=';', encoding='latin-1', on_bad_lines='skip')

# Cast Year-Of-Publication to string so PyArrow doesn't fail on mixed raw text/numbers
books_df['Year-Of-Publication'] = books_df['Year-Of-Publication'].astype(str)

# 2. Add ingestion timestamp
now = pd.Timestamp.now()
users_df['ingestion_timestamp'] = now
books_df['ingestion_timestamp'] = now
ratings_df['ingestion_timestamp'] = now

# 3. Save untouched tables to bronze/ as Parquet
users_df.to_parquet('bronze/users.parquet', index=False)
books_df.to_parquet('bronze/books.parquet', index=False)
ratings_df.to_parquet('bronze/ratings.parquet', index=False)

print("Bronze layer saved successfully!")

/tmp/ipykernel_2257/3134578171.py:5: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  books_df = pd.read_csv('BX-Books.csv', sep=';', encoding='latin-1', on_bad_lines='skip')


Bronze layer saved successfully!


In [12]:
import pandas as pd

# Read Bronze Parquet tables back
users = pd.read_parquet('bronze/users.parquet')
books = pd.read_parquet('bronze/books.parquet')
ratings = pd.read_parquet('bronze/ratings.parquet')

# 1. Shape, data types, and missing-value count
for name, df in [('Users', users), ('Books', books), ('Ratings', ratings)]:
    print(f"=== {name} ===")
    print("Shape:", df.shape)
    print(df.dtypes)
    print("Missing values:\n", df.isnull().sum(), "\n")

# 2. Distribution of ratings (0 vs 1-10)
print("=== Rating Distribution ===")
print(ratings['Book-Rating'].value_counts().sort_index())
zero_count = (ratings['Book-Rating'] == 0).sum()
print(f"Zero ratings: {zero_count} ({zero_count / len(ratings) * 100:.2f}%)\n")

# 3. Distribution of user ages and publication years
print("=== User Age Summary & Impossible Values ===")
print(users['Age'].describe())
print("Ages < 5 or > 100:", users[(users['Age'] < 5) | (users['Age'] > 100)]['Age'].count())

print("\n=== Publication Year Summary & Impossible Values ===")
pub_years = pd.to_numeric(books['Year-Of-Publication'], errors='coerce')
print("Non-numeric years:", pub_years.isnull().sum())
print("Year == 0:", (pub_years == 0).sum())
print("Year > 2026:", (pub_years > 2026).sum())
print("Sample extreme years:", pub_years[(pub_years == 0) | (pub_years > 2026)].value_counts().head(10), "\n")

# 4. Number of ratings per user and per book
print("=== Activity Distribution ===")
user_counts = ratings['User-ID'].value_counts()
book_counts = ratings['ISBN'].value_counts()
print("Ratings per User quantiles:\n", user_counts.quantile([0.25, 0.5, 0.75, 0.9, 0.99]))
print("Ratings per Book quantiles:\n", book_counts.quantile([0.25, 0.5, 0.75, 0.9, 0.99]), "\n")

# 5. Matrix sparsity
n_users = ratings['User-ID'].nunique()
n_books = ratings['ISBN'].nunique()
total_ratings = len(ratings)
density = total_ratings / (n_users * n_books)
sparsity = (1 - density) * 100
print("=== Matrix Sparsity ===")
print(f"Unique Users: {n_users}, Unique Books: {n_books}, Total Ratings: {total_ratings}")
print(f"Filled cells: {density * 100:.6f}% | Matrix Sparsity: {sparsity:.6f}%")

=== Users ===
Shape: (278858, 4)
User-ID                         int64
Location                       object
Age                           float64
ingestion_timestamp    datetime64[us]
dtype: object
Missing values:
 User-ID                     0
Location                    0
Age                    110762
ingestion_timestamp         0
dtype: int64 

=== Books ===
Shape: (271360, 9)
ISBN                           object
Book-Title                     object
Book-Author                    object
Year-Of-Publication            object
Publisher                      object
Image-URL-S                    object
Image-URL-M                    object
Image-URL-L                    object
ingestion_timestamp    datetime64[us]
dtype: object
Missing values:
 ISBN                   0
Book-Title             0
Book-Author            2
Year-Of-Publication    0
Publisher              2
Image-URL-S            0
Image-URL-M            0
Image-URL-L            3
ingestion_timestamp    0
dtype: int64 

===

### Step 2: Bronze EDA Findings & Silver Layer Plan
1. **Shape, Data Types & Missing Values:** `Users` has ~278k rows (with ~110k missing `Age` values), `Books` has ~271k rows (`Year-Of-Publication` stored as `object` due to text errors, plus minor missing author/publisher fields), and `Ratings` has ~1.14M rows with no nulls; in Silver, we will enforce proper numeric types, impute missing ages, and clean/drop unneeded columns.
2. **Rating Distribution:** Over 62% (~716k) of all entries in `Ratings` have a `Book-Rating` of `0` (implicit interactions rather than explicit scores); in Silver, we will drop all `0` ratings and retain only explicit ratings from `1` to `10`.
3. **User Ages & Publication Years:** `Age` contains impossible outliers (`0` up to `244`), and `Year-Of-Publication` contains `0`, future years (e.g., `2030`, `2050`), and shifted text strings; in Silver, we will set ages `< 5` or `> 100` to null and fill with the median age, and coerce publication years while setting `0` or years `> 2026` to null.
4. **Ratings per User & Book:** Both distributions are heavily right-skewed, with the median user and median book having only 1–2 ratings; in Silver, we will filter out cold-start users and books below a minimum rating threshold (e.g., `>= 10` ratings) to reduce noise and matrix size.
5. **Matrix Sparsity:** Over `99.99%` of possible user-book pairs are empty (fewer than `0.01%` have a rating); in Silver and Gold, we will filter inactive users/books and store the rating matrix in a `scipy.sparse` CSR format instead of a dense array.

In [13]:
import pandas as pd
import numpy as np

# Load Bronze tables
users = pd.read_parquet('bronze/users.parquet')
books = pd.read_parquet('bronze/books.parquet')
ratings = pd.read_parquet('bronze/ratings.parquet')

# 1. Remove duplicate rows
users = users.drop_duplicates(subset=['User-ID']).copy()
books = books.drop_duplicates(subset=['ISBN']).copy()
ratings = ratings.drop_duplicates(subset=['User-ID', 'ISBN']).copy()

# 2. Remove ratings of 0 and keep explicit ratings from 1 to 10
ratings = ratings[(ratings['Book-Rating'] >= 1) & (ratings['Book-Rating'] <= 10)].copy()

# 3. Fix user ages: < 5 or > 100 -> NaN, then fill with median age
users.loc[(users['Age'] < 5) | (users['Age'] > 100), 'Age'] = np.nan
median_age = users['Age'].median()
users['Age'] = users['Age'].fillna(median_age)

# 4. Split Location text into city, state, and country columns
loc_splits = users['Location'].fillna('').str.split(',', n=2, expand=True)
users['city'] = loc_splits[0].str.strip().str.lower()
users['state'] = loc_splits[1].fillna('').str.strip().str.lower()
users['country'] = loc_splits[2].fillna('').str.strip().str.lower()
users = users.drop(columns=['Location'])

# 5. Fix publication years: set 0 or future years (> 2026) to null
books['Year-Of-Publication'] = pd.to_numeric(books['Year-Of-Publication'], errors='coerce')
books.loc[(books['Year-Of-Publication'] <= 0) | (books['Year-Of-Publication'] > 2026), 'Year-Of-Publication'] = np.nan

# 6. Clean text columns: strip extra spaces, standardise case, and drop image-URL columns
img_cols = [c for c in books.columns if 'Image-URL' in c]
books = books.drop(columns=img_cols)

for col in ['Book-Title', 'Book-Author', 'Publisher']:
    books[col] = books[col].fillna('unknown').astype(str).str.strip().str.title()

books['ISBN'] = books['ISBN'].astype(str).str.strip().str.upper()
ratings['ISBN'] = ratings['ISBN'].astype(str).str.strip().str.upper()

# 7. Keep only ratings whose book and user exist in books and users tables
ratings = ratings[ratings['User-ID'].isin(users['User-ID']) & ratings['ISBN'].isin(books['ISBN'])].copy()

# 8. Keep only users and books with at least MIN_RATINGS = 10
MIN_RATINGS = 10
while True:
    valid_users = ratings['User-ID'].value_counts()
    valid_users = valid_users[valid_users >= MIN_RATINGS].index

    valid_books = ratings['ISBN'].value_counts()
    valid_books = valid_books[valid_books >= MIN_RATINGS].index

    filtered = ratings[ratings['User-ID'].isin(valid_users) & ratings['ISBN'].isin(valid_books)]
    if len(filtered) == len(ratings):
        break
    ratings = filtered.copy()

# Filter users and books tables to match the active ratings set
users_silver = users[users['User-ID'].isin(ratings['User-ID'])].copy()
books_silver = books[books['ISBN'].isin(ratings['ISBN'])].copy()

# Save cleaned tables to silver/
users_silver.to_parquet('silver/users.parquet', index=False)
books_silver.to_parquet('silver/books.parquet', index=False)
ratings.to_parquet('silver/ratings.parquet', index=False)

print(f"Silver layer saved! Users: {len(users_silver)}, Books: {len(books_silver)}, Ratings: {len(ratings)}")

Silver layer saved! Users: 1820, Books: 2087, Ratings: 41469


### Step 3: Minimum Rating Threshold Justification
We selected a minimum rating threshold of **10 ratings per user and per book**. In our Step 2 EDA, the rating distributions were heavily right-skewed—the median user and median book had only 1–2 ratings, resulting in over 99.99% matrix sparsity. Filtering out users and books with fewer than 10 explicit ratings eliminates cold-start noise, prevents unreliable similarity scores from single-rating overlaps, and keeps the interaction matrix compact enough for fast item-based cosine similarity computation.

In [14]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix, save_npz

# Load Silver tables
users_silver = pd.read_parquet('silver/users.parquet')
books_silver = pd.read_parquet('silver/books.parquet')
ratings_silver = pd.read_parquet('silver/ratings.parquet')

# 1. Create contiguous integer indices for users and books
unique_users = sorted(ratings_silver['User-ID'].unique())
unique_books = sorted(ratings_silver['ISBN'].unique())

user_to_idx = {uid: i for i, uid in enumerate(unique_users)}
book_to_idx = {isbn: i for i, isbn in enumerate(unique_books)}

ratings_silver['user_idx'] = ratings_silver['User-ID'].map(user_to_idx)
ratings_silver['book_idx'] = ratings_silver['ISBN'].map(book_to_idx)

# Build and save sparse user-book rating matrix (rows = users, cols = books)
rating_matrix = csr_matrix(
    (ratings_silver['Book-Rating'].astype(np.float32),
     (ratings_silver['user_idx'], ratings_silver['book_idx'])),
    shape=(len(unique_users), len(unique_books))
)
save_npz('gold/user_book_matrix.npz', rating_matrix)

# Save user index mapping for Step 5 & 6 lookups
user_lookup = pd.DataFrame({
    'user_idx': range(len(unique_users)),
    'User-ID': unique_users
})
user_lookup.to_parquet('gold/user_lookup.parquet', index=False)

# 2. Book lookup table (book_idx, ISBN, Book-Title, Book-Author)
book_lookup = books_silver[books_silver['ISBN'].isin(book_to_idx)][['ISBN', 'Book-Title', 'Book-Author']].copy()
book_lookup['book_idx'] = book_lookup['ISBN'].map(book_to_idx)
book_lookup = book_lookup.sort_values('book_idx').reset_index(drop=True)
book_lookup.to_parquet('gold/book_lookup.parquet', index=False)

# 3. User-level and book-level aggregates (average rating and number of ratings)
user_agg = ratings_silver.groupby('User-ID').agg(
    rating_count=('Book-Rating', 'count'),
    avg_rating=('Book-Rating', 'mean')
).reset_index()
user_agg.to_parquet('gold/user_aggregates.parquet', index=False)

book_agg = ratings_silver.groupby('ISBN').agg(
    rating_count=('Book-Rating', 'count'),
    avg_rating=('Book-Rating', 'mean')
).reset_index()
book_agg.to_parquet('gold/book_aggregates.parquet', index=False)

print("Gold layer saved successfully!")
print(f"Sparse Matrix Shape (Users x Books): {rating_matrix.shape}, Non-zero entries: {rating_matrix.nnz}")

Gold layer saved successfully!
Sparse Matrix Shape (Users x Books): (1820, 2030), Non-zero entries: 41469


In [15]:
import pandas as pd
import numpy as np
from scipy.sparse import load_npz
from sklearn.neighbors import NearestNeighbors

# 1. Load Gold artifacts and orient matrix so rows = books, columns = users
user_book_matrix = load_npz('gold/user_book_matrix.npz')
book_user_matrix = user_book_matrix.T.tocsr()

book_lookup = pd.read_parquet('gold/book_lookup.parquet').drop_duplicates(subset=['book_idx'])
user_lookup = pd.read_parquet('gold/user_lookup.parquet').drop_duplicates(subset=['user_idx'])

user_id_to_idx = dict(zip(user_lookup['User-ID'], user_lookup['user_idx']))
idx_to_book = book_lookup.set_index('book_idx')[['Book-Title', 'Book-Author']].to_dict('index')

# 2. Compute cosine similarity between books
nn_model = NearestNeighbors(metric='cosine', algorithm='brute')
nn_model.fit(book_user_matrix)

# 3. Function returning the 10 most similar books for a given book title
def get_similar_books(book_title, n=10):
    matches = book_lookup[book_lookup['Book-Title'].str.lower() == book_title.strip().lower()]
    if matches.empty:
        matches = book_lookup[book_lookup['Book-Title'].str.lower().str.contains(book_title.strip().lower(), regex=False)]
    if matches.empty:
        return f"Book '{book_title}' not found in Gold lookup."

    book_idx = matches.iloc[0]['book_idx']
    matched_title = matches.iloc[0]['Book-Title']

    n_query = min(n + 1, book_user_matrix.shape[0])
    distances, indices = nn_model.kneighbors(book_user_matrix[book_idx], n_neighbors=n_query)

    results = []
    for dist, idx in zip(distances[0], indices[0]):
        if idx == book_idx:
            continue
        info = idx_to_book[idx]
        results.append({
            'Book-Title': info['Book-Title'],
            'Book-Author': info['Book-Author'],
            'Similarity': round(float(1 - dist), 4)
        })
        if len(results) == n:
            break

    print(f"Books most similar to '{matched_title}':")
    return pd.DataFrame(results)

# 4. User recommendation function
def recommend(user_id, n=10):
    if user_id not in user_id_to_idx:
        return f"User-ID {user_id} not found in Gold dataset."

    u_idx = user_id_to_idx[user_id]
    user_row = user_book_matrix[u_idx]

    rated_book_indices = user_row.indices
    rated_scores = user_row.data
    already_read = set(rated_book_indices)

    # Filter for books the user rated highly (>= 7, or above their average if they rate strictly)
    high_mask = rated_scores >= 7
    if not np.any(high_mask):
        high_mask = rated_scores >= np.mean(rated_scores)

    liked_books = rated_book_indices[high_mask]
    liked_ratings = rated_scores[high_mask]

    candidate_scores = {}
    n_neighbors = min(20, book_user_matrix.shape[0])

    for b_idx, u_rating in zip(liked_books, liked_ratings):
        distances, indices = nn_model.kneighbors(book_user_matrix[b_idx], n_neighbors=n_neighbors)
        for dist, neighbor_idx in zip(distances[0], indices[0]):
            if neighbor_idx in already_read:
                continue
            similarity = 1.0 - float(dist)
            if similarity > 0:
                candidate_scores[neighbor_idx] = candidate_scores.get(neighbor_idx, 0.0) + (similarity * float(u_rating))

    ranked_candidates = sorted(candidate_scores.items(), key=lambda x: x[1], reverse=True)[:n]

    recs = []
    for b_idx, score in ranked_candidates:
        info = idx_to_book[b_idx]
        recs.append({
            'Book-Title': info['Book-Title'],
            'Book-Author': info['Book-Author'],
            'Weighted-Score': round(score, 4)
        })
    return pd.DataFrame(recs)

print("Recommender functions ready!")

Recommender functions ready!


In [16]:
# 1. Test item-to-item similarity for a sample book
sample_title = book_lookup['Book-Title'].iloc[0]
display(get_similar_books(sample_title, n=10))
print("\n" + "="*80 + "\n")

# 2. Run recommend() for 3 different users from the Gold dataset
sample_users = user_lookup['User-ID'].iloc[[0, 10, 25]].tolist()

for uid in sample_users:
    u_idx = user_id_to_idx[uid]
    user_row = user_book_matrix[u_idx]

    # Collect books this user rated highly
    liked = []
    for b_idx, rating in zip(user_row.indices, user_row.data):
        if rating >= 7:
            info = idx_to_book[b_idx]
            liked.append({
                'Book-Title': info['Book-Title'],
                'Book-Author': info['Book-Author'],
                'User-Rating': rating
            })
    liked_df = pd.DataFrame(liked).sort_values('User-Rating', ascending=False).head(10)

    recs_df = recommend(uid, n=10)

    print(f"==================== USER ID: {uid} ====================")
    print("\n--- Books User Rated Highly (Top 10) ---")
    display(liked_df)
    print("\n--- Top 10 Recommended Books ---")
    display(recs_df)
    print("\n")

Books most similar to 'Great Gatsby (Reissue)':


,Book-Title,Book-Author,Similarity
0,Name Of The Rose-Nla,Umberto Eco,0.2699
1,The Cereal Murders,Diane Mott Davidson,0.1974
2,"The Subtle Knife (His Dark Materials, Book 2)",Philip Pullman,0.1898
3,The Little Prince,Antoine De Saint-Exupã©Ry,0.1499
4,The Outsiders,S. E. Hinton,0.1479
5,Devil'S Waltz (Alex Delaware Novels (Paperback)),Jonathan Kellerman,0.1370
6,Fast Food Nation: The Dark Side Of The All-Ame...,Eric Schlosser,0.1338
7,The Crucible: A Play In Four Acts (Penguin Plays),Arthur Miller,0.1330
8,The Book Borrower: A Novel,Alice Mattison,0.1317
9,A Widow For One Year,John Irving,0.1278




==================== USER ID: 243 ====================

--- Books User Rated Highly (Top 10) ---


,Book-Title,Book-Author,User-Rating
0,The Bean Trees,Barbara Kingsolver,10.0
5,Memoirs Of A Geisha,Arthur Golden,10.0
9,The General'S Daughter,Nelson Demille,9.0
2,The Pilot'S Wife : A Novel,Anita Shreve,9.0
3,Me Talk Pretty One Day,David Sedaris,9.0
7,Unnatural Exposure,Patricia Daniels Cornwell,9.0
1,The God Of Small Things,Arundhati Roy,7.0
4,"River, Cross My Heart",Breena Clarke,7.0
6,A Map Of The World,Jane Hamilton,7.0
8,A Painted House,John Grisham,7.0



--- Top 10 Recommended Books ---


,Book-Title,Book-Author,Weighted-Score
0,Vinegar Hill (Oprah'S Book Club (Paperback)),A. Manette Ansay,4.6791
1,The Rapture Of Canaan,Sheri Reynolds,4.2899
2,We Were The Mulvaneys,Joyce Carol Oates,3.5926
3,Drowning Ruth (Oprah'S Book Club),Christina Schwarz,3.5790
4,The Body Farm,Patricia Daniels Cornwell,3.5751
5,From Potter'S Field,Patricia Daniels Cornwell,3.3384
6,Fall On Your Knees (Oprah #45),Ann-Marie Macdonald,3.3057
7,Animal Dreams,Barbara Kingsolver,3.2310
8,Point Of Origin,Patricia Daniels Cornwell,3.0242
9,Jewel,Bret Lott,2.9714


... (4 outputs left out)



==================== USER ID: 5582 ====================

--- Books User Rated Highly (Top 10) ---


,Book-Title,Book-Author,User-Rating
0,Bridget Jones'S Diary,Helen Fielding,10.0
2,Four To Score (A Stephanie Plum Novel),Janet Evanovich,10.0
3,Hot Six : A Stephanie Plum Novel (A Stephanie ...,Janet Evanovich,10.0
33,When I Am An Old Woman I Shall Wear Purple,Sandra Martz,10.0
6,"Memnoch The Devil (Vampire Chronicles, No 5)",Anne Rice,10.0
7,A Kiss Of Shadows (Meredith Gentry Novels (Pap...,Laurell K. Hamilton,10.0
17,Narcissus In Chains (Anita Blake Vampire Hunte...,Laurell K. Hamilton,10.0
15,Betrayal In Death,Nora Roberts,10.0
14,Judgment In Death,J. D. Robb,10.0
13,Witness In Death (Eve Dallas Mysteries (Paperb...,J. D. Robb,10.0



--- Top 10 Recommended Books ---


,Book-Title,Book-Author,Weighted-Score
0,Purity In Death,J.D. Robb,22.6678
1,Conspiracy In Death,J. D. Robb,19.4072
2,The Lunatic Cafe (Anita Blake Vampire Hunter (...,Laurell K. Hamilton,19.1790
3,Circus Of The Damned (Anita Blake Vampire Hunt...,Laurell K. Hamilton,18.1044
4,Naked In Death,J. D. Robb,17.6996
5,Bloody Bones (Anita Blake Vampire Hunter (Pape...,Laurell K. Hamilton,17.4787
6,A Caress Of Twilight (Meredith Gentry Novels (...,Laurell K. Hamilton,16.1464
7,The Laughing Corpse (Anita Blake Vampire Hunte...,Laurell K. Hamilton,15.6620
8,Killing Dance (Anita Blake Vampire Hunter (Pap...,Laurell K. Hamilton,15.1043
9,Imitation In Death (Eve Dallas Mysteries (Pape...,J. D. Robb,14.6707
